# Elyra Visual Pipeline with Model Registry — RHOAI Demo

**~15 minutes** | Build a visual ML pipeline with Elyra, train a model, register it in Model Registry.

> Each step below is self-contained — save it as a `.py` file and drag it into Elyra's visual pipeline editor.

## What is Elyra?

**Elyra** is a JupyterLab extension that lets you build ML pipelines visually — drag & drop notebooks or Python scripts as pipeline nodes, connect them, and submit to a Kubeflow Pipelines backend.

In RHOAI, Elyra is pre-installed in the **Standard Data Science** and **CUDA** workbench images. You get:
- A **visual pipeline editor** (`.pipeline` files)
- Each node runs as an isolated container on the pipeline server
- Automatic S3 artifact passing between nodes
- Pipeline submission directly from JupyterLab

This notebook contains the three scripts that form the pipeline, plus instructions on wiring them up in Elyra.

---
## Setup

In [ ]:
!pip install -q boto3 scikit-learn requests

In [ ]:
import os
import subprocess

# --- MinIO ---
MINIO_ENDPOINT = "http://minio-service.minio.svc.cluster.local:9000"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "pipelines"

# --- Model Registry ---
# From inside the cluster (workbench):
MODEL_REGISTRY_URL = "https://demo-registry.rhoai-model-registries.svc:8443"
# From outside the cluster, use:
# MODEL_REGISTRY_URL = "https://demo-registry-rest.apps.ocp.xlwsd.sandbox1213.opentlc.com"
MODEL_REGISTRY_API = f"{MODEL_REGISTRY_URL}/api/model_registry/v1alpha3"

# Auth token — from mounted SA or oc CLI
try:
    with open("/var/run/secrets/kubernetes.io/serviceaccount/token") as f:
        AUTH_TOKEN = f.read().strip()
except FileNotFoundError:
    AUTH_TOKEN = subprocess.check_output(["oc", "whoami", "-t"]).decode().strip()

print(f"MinIO: {MINIO_ENDPOINT}")
print(f"Registry: {MODEL_REGISTRY_API}")
print(f"Token: {AUTH_TOKEN[:20]}...")

---
## Step 1 — Data Preparation

Generates a synthetic classification dataset and uploads train/test splits to MinIO.

> **Elyra node**: Save this cell as `1_data_prep.py`

In [ ]:
# === 1_data_prep.py — Self-contained Elyra pipeline node ===
import io
import boto3
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

MINIO_ENDPOINT = "http://minio-service.minio.svc.cluster.local:9000"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "pipelines"

# Generate synthetic data
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42
)
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

X_train, X_test, y_train, y_test = train_test_split(
    df.drop("target", axis=1), df["target"], test_size=0.2, random_state=42
)

train_df = X_train.copy()
train_df["target"] = y_train
test_df = X_test.copy()
test_df["target"] = y_test

# Upload to MinIO
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
)

for name, data in [("data/train.csv", train_df), ("data/test.csv", test_df)]:
    buf = io.BytesIO()
    data.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=MINIO_BUCKET, Key=name, Body=buf.getvalue())
    print(f"Uploaded {name} ({len(data)} rows)")

print("\nData preparation complete.")

---
## Step 2 — Model Training

Loads training data from MinIO, trains a GradientBoostingClassifier, evaluates it, and saves the model pickle back to MinIO.

> **Elyra node**: Save this cell as `2_train_model.py`

In [ ]:
# === 2_train_model.py — Self-contained Elyra pipeline node ===
import io
import pickle
import boto3
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score

MINIO_ENDPOINT = "http://minio-service.minio.svc.cluster.local:9000"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "pipelines"

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
)

# Load data from MinIO
train_obj = s3.get_object(Bucket=MINIO_BUCKET, Key="data/train.csv")
train_df = pd.read_csv(io.BytesIO(train_obj["Body"].read()))

test_obj = s3.get_object(Bucket=MINIO_BUCKET, Key="data/test.csv")
test_df = pd.read_csv(io.BytesIO(test_obj["Body"].read()))

X_train = train_df.drop("target", axis=1)
y_train = train_df["target"]
X_test = test_df.drop("target", axis=1)
y_test = test_df["target"]

# Train
model = GradientBoostingClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

# Save model to MinIO
model_bytes = pickle.dumps(model)
s3.put_object(Bucket=MINIO_BUCKET, Key="models/gradient_boosting.pkl", Body=model_bytes)
print(f"\nModel saved to s3://{MINIO_BUCKET}/models/gradient_boosting.pkl ({len(model_bytes)} bytes)")

# Write metrics to a file for downstream nodes (Elyra artifact passing)
with open("metrics.txt", "w") as f:
    f.write(f"accuracy={accuracy:.4f}\nf1={f1:.4f}\n")
print("Metrics written to metrics.txt")

---
## Step 3 — Register in Model Registry

Registers the trained model in the RHOAI Model Registry using the REST API.

> **Elyra node**: Save this cell as `3_register_model.py`

In [ ]:
# === 3_register_model.py — Self-contained Elyra pipeline node ===
import os
import json
import subprocess
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODEL_REGISTRY_URL = "https://demo-registry.rhoai-model-registries.svc:8443"
MODEL_REGISTRY_API = f"{MODEL_REGISTRY_URL}/api/model_registry/v1alpha3"

try:
    with open("/var/run/secrets/kubernetes.io/serviceaccount/token") as f:
        AUTH_TOKEN = f.read().strip()
except FileNotFoundError:
    AUTH_TOKEN = subprocess.check_output(["oc", "whoami", "-t"]).decode().strip()

headers = {
    "Authorization": f"Bearer {AUTH_TOKEN}",
    "Content-Type": "application/json",
}

# Read metrics from upstream node (if available)
accuracy = "unknown"
f1 = "unknown"
if os.path.exists("metrics.txt"):
    with open("metrics.txt") as f:
        for line in f:
            k, v = line.strip().split("=")
            if k == "accuracy":
                accuracy = v
            elif k == "f1":
                f1 = v

# 1. Register the model
resp = requests.post(
    f"{MODEL_REGISTRY_API}/registered_models",
    headers=headers,
    json={
        "name": "elyra-gradient-boosting",
        "description": "GradientBoostingClassifier trained via Elyra visual pipeline",
        "customProperties": {
            "pipeline": {"stringValue": "elyra-visual"},
            "framework": {"stringValue": "scikit-learn"},
        },
    },
    verify=False,
)
resp.raise_for_status()
registered_model = resp.json()
model_id = registered_model["id"]
print(f"Registered model: {registered_model['name']} (id={model_id})")

# 2. Create model version
resp = requests.post(
    f"{MODEL_REGISTRY_API}/registered_models/{model_id}/versions",
    headers=headers,
    json={
        "name": "v1",
        "description": f"accuracy={accuracy}, f1={f1}",
        "customProperties": {
            "accuracy": {"stringValue": str(accuracy)},
            "f1_score": {"stringValue": str(f1)},
            "source": {"stringValue": "elyra-pipeline"},
        },
    },
    verify=False,
)
resp.raise_for_status()
model_version = resp.json()
version_id = model_version["id"]
print(f"Created version: {model_version['name']} (id={version_id})")

# 3. Create model artifact
resp = requests.post(
    f"{MODEL_REGISTRY_API}/model_versions/{version_id}/artifacts",
    headers=headers,
    json={
        "name": "gradient-boosting-pkl",
        "description": "Pickle-serialized GradientBoostingClassifier",
        "uri": "s3://pipelines/models/gradient_boosting.pkl",
        "artifactType": "model-artifact",
        "modelFormatName": "sklearn",
        "modelFormatVersion": "1.5",
    },
    verify=False,
)
resp.raise_for_status()
artifact = resp.json()
print(f"Created artifact: {artifact.get('name', 'ok')} -> s3://pipelines/models/gradient_boosting.pkl")
print("\nModel registered successfully in Model Registry!")

---
## How to Use with Elyra Visual Editor

### Pre-built files included
The repo includes everything you need — no copy-pasting required:
- `1_data_prep.py` — data generation node
- `2_train_model.py` — model training node
- `3_register_model.py` — model registry node
- `elyra-model-registry.pipeline` — pre-configured pipeline file

### Open the pipeline
1. In JupyterLab, navigate to `runbooks/`
2. Double-click `elyra-model-registry.pipeline` — it opens in the visual editor
3. You'll see three nodes connected: **Data Preparation → Train Model → Register in Model Registry**

### Configure the runtime
Before running, you need a runtime configuration:
1. Click **Runtimes** in the left sidebar (puzzle piece icon)
2. Add a **Kubeflow Pipelines** runtime:
   - **Kubeflow Pipelines API Endpoint**: `https://ds-pipeline-dspa-rhoai-playground.apps.ocp.xlwsd.sandbox1213.opentlc.com`
   - **Cloud Object Storage Endpoint**: `http://minio-service.minio.svc.cluster.local:9000`
   - **Bucket**: `pipelines`
   - **Access Key**: `minio`
   - **Secret Key**: `minio123`

### Configure each node
- Right-click a node → **Properties**
- Set the **Runtime Image** to the standard RHOAI Python image
- pip packages are pre-configured in the pipeline file

### Run the pipeline
- Click the **Run** button (play icon) in the toolbar
- The pipeline submits to the pipeline server
- Monitor in the RHOAI dashboard: **Pipelines > Runs**

---
## Run All Steps Sequentially (No Elyra)

If Elyra isn't available, run the pipeline steps directly in this notebook:

In [ ]:
print("=" * 60)
print("STEP 1: Data Preparation")
print("=" * 60)
# Re-run the data prep cell above (In[5])
# ... already executed if you ran cells in order

print()
print("=" * 60)
print("STEP 2: Model Training")
print("=" * 60)
# ... already executed if you ran cells in order

print()
print("=" * 60)
print("STEP 3: Register in Model Registry")
print("=" * 60)
# ... already executed if you ran cells in order

print()
print("All steps complete — check the Model Registry in the RHOAI dashboard.")
print("Navigate to: Model resources and operations > Model registry > demo-registry")

---
## Verify in Model Registry

In [ ]:
import requests
import json
import subprocess
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODEL_REGISTRY_URL = "https://demo-registry.rhoai-model-registries.svc:8443"
MODEL_REGISTRY_API = f"{MODEL_REGISTRY_URL}/api/model_registry/v1alpha3"

try:
    with open("/var/run/secrets/kubernetes.io/serviceaccount/token") as f:
        AUTH_TOKEN = f.read().strip()
except FileNotFoundError:
    AUTH_TOKEN = subprocess.check_output(["oc", "whoami", "-t"]).decode().strip()

headers = {"Authorization": f"Bearer {AUTH_TOKEN}"}

resp = requests.get(
    f"{MODEL_REGISTRY_API}/registered_models",
    headers=headers,
    verify=False,
)
resp.raise_for_status()
models = resp.json()

print(f"Registered models ({models['size']}):")
for m in models.get("items", []):
    print(f"  - {m['name']} (id={m['id']})")
    # Get versions
    v_resp = requests.get(
        f"{MODEL_REGISTRY_API}/registered_models/{m['id']}/versions",
        headers=headers,
        verify=False,
    )
    if v_resp.ok:
        for v in v_resp.json().get("items", []):
            print(f"    version: {v['name']} — {v.get('description', '')}")

---
## Key Takeaways

| Feature | Benefit |
|---------|--------|
| **Visual editor** | No code for pipeline orchestration — drag, connect, run |
| **Self-contained nodes** | Each script is independently testable |
| **S3 artifacts** | Data flows between nodes via object storage |
| **Model Registry** | Every trained model is versioned and discoverable |
| **Pipeline server** | Runs on Kubernetes — scalable, auditable, repeatable |